# 01 Data Cleaning and Exploratory Analysis

This notebook loads Online Retail II, standardizes fields, handles cancellations/returns, and creates dashboard-ready sales KPI tables.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_PATH = Path("../data/raw/online_retail_II.xlsx")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

In [ ]:
# Load both yearly worksheets
xl = pd.ExcelFile(RAW_PATH)
frames = []
for sheet in xl.sheet_names:
    temp = pd.read_excel(RAW_PATH, sheet_name=sheet)
    temp["source_sheet"] = sheet
    frames.append(temp)

raw = pd.concat(frames, ignore_index=True)
print(raw.shape)
raw.head()

In [ ]:
# Standardize columns
rename_map = {
    "Invoice": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
}

df = raw.rename(columns=rename_map).copy()
df["invoice_no"] = df["invoice_no"].astype(str)
df["stock_code"] = df["stock_code"].astype(str)
df["description"] = df["description"].astype(str).str.strip()
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce")

df["revenue"] = df["quantity"] * df["unit_price"]
df["is_cancelled"] = df["invoice_no"].str.startswith("C") | (df["quantity"] < 0)
df["invoice_month"] = df["invoice_date"].dt.to_period("M").astype(str)
df["year"] = df["invoice_date"].dt.year

df.head()

In [ ]:
# Data quality summary
quality_summary = pd.DataFrame({
    "metric": [
        "rows", "date_min", "date_max", "unique_invoices", "unique_customers",
        "unique_products", "unique_countries", "missing_customer_id_rows",
        "cancelled_or_return_rows", "non_positive_price_rows"
    ],
    "value": [
        len(df), df["invoice_date"].min(), df["invoice_date"].max(),
        df["invoice_no"].nunique(), df["customer_id"].nunique(dropna=True),
        df["stock_code"].nunique(), df["country"].nunique(),
        df["customer_id"].isna().sum(), df["is_cancelled"].sum(),
        (df["unit_price"] <= 0).sum()
    ]
})
quality_summary

In [ ]:
# Valid sales transactions for KPI analysis
sales = df[(~df["is_cancelled"]) & (df["quantity"] > 0) & (df["unit_price"] > 0)].copy()

monthly_sales = sales.groupby("invoice_month").agg(
    total_revenue=("revenue", "sum"),
    orders=("invoice_no", "nunique"),
    active_customers=("customer_id", lambda x: x.nunique(dropna=True)),
    units_sold=("quantity", "sum")
).reset_index()
monthly_sales["avg_order_value"] = monthly_sales["total_revenue"] / monthly_sales["orders"]
monthly_sales.head()

In [ ]:
# Country and product performance tables
country_performance = sales.groupby("country").agg(
    total_revenue=("revenue", "sum"),
    orders=("invoice_no", "nunique"),
    customers=("customer_id", lambda x: x.nunique(dropna=True)),
    units_sold=("quantity", "sum")
).reset_index().sort_values("total_revenue", ascending=False)
country_performance["avg_order_value"] = country_performance["total_revenue"] / country_performance["orders"]

product_performance = sales.groupby(["stock_code", "description"]).agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum"),
    orders=("invoice_no", "nunique"),
    customers=("customer_id", lambda x: x.nunique(dropna=True))
).reset_index().sort_values("total_revenue", ascending=False)

country_performance.head(10), product_performance.head(10)

In [ ]:
# Save processed outputs
quality_summary.to_csv(PROCESSED_DIR / "data_quality_summary.csv", index=False)
monthly_sales.to_csv(PROCESSED_DIR / "monthly_sales_summary.csv", index=False)
country_performance.to_csv(PROCESSED_DIR / "country_performance.csv", index=False)
product_performance.head(200).to_csv(PROCESSED_DIR / "product_performance_top200.csv", index=False)

# Save cleaned transaction sample for GitHub preview. Do not upload the full raw dataset if it is too large.
sales.head(10000).to_csv(PROCESSED_DIR / "cleaned_sales_sample_10000.csv", index=False)

print("Saved processed sales outputs.")

In [ ]:
# Quick monthly revenue chart
plt.figure(figsize=(12, 5))
plt.plot(monthly_sales["invoice_month"], monthly_sales["total_revenue"])
plt.xticks(rotation=75)
plt.title("Monthly Revenue Trend")
plt.xlabel("Invoice Month")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()